> Implementation update: the production model library, bounded hybrid adaptation and empirical interval-calibration APIs are now implemented. See [HYBRID_AND_UNCERTAINTY.md](HYBRID_AND_UNCERTAINTY.md) and [notebook 03](03_hybrid_adaptation_and_uncertainty.ipynb). The numerical lessons below are baseline examples, not customer validation.

# 02 · Run the pharmaceutical model library

**16 additional reduced models, plus reuse of existing models.** Execute every family today, then inspect any model in depth. No sequential lesson prerequisite.

All new input values are authored synthetic assumptions. Numerical checks establish code behaviour, not experimental accuracy, process safety or commercial qualification. These are reduced process models rather than complete plant digital twins.

Read [LIBRARY.md](LIBRARY.md) for scope and [CATALOG.md](CATALOG.md) for equations, required inputs and units.

**Next:** neural hybrid terms, site adaptation and interval calibration are covered in [notebook 03](03_hybrid_adaptation_and_uncertainty.ipynb) and [HYBRID_AND_UNCERTAINTY.md](HYBRID_AND_UNCERTAINTY.md).

In [ ]:
from pathlib import Path
import sys, json
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "backend/src/lifesci/process_models/registry.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook inside the patched BioDemo checkout")
sys.path.insert(0, str(root / "backend"))
from src.lifesci.process_models.registry import models, get, existing_model_inventory
from src.lifesci.process_models.examples import examples
from src.lifesci.process_models.analysis import fit_parameter, compare, parameter_scenarios
from src.lifesci.process_models.flowsheets import purification_train
cases = examples()
print("Additional models:", len(models()))
print("Reused existing modules:", existing_model_inventory())


## 1. Run all model families

Each calculation uses a strict input contract. Inspect `inputs`, `equation`, `model_kind` and `limitations` before interpreting a result. A successful calculation does not establish that its assumptions fit a real product or site.


In [ ]:
results = {mid: get(mid).run(config) for mid, config in cases.items()}
for mid, result in results.items():
    print(mid, result["status"], result["model_kind"], result["violations"])


## 2. Inspect the curves together

Every panel is a separate model/example; no causal connection between panels is assumed. Note that compaction uses pressure, not time. Equilibrium capture and mixing-power checks are scalar screens.


In [ ]:
fig, axes = plt.subplots(4,4,figsize=(16,12))
for ax, (mid, result) in zip(axes.flat, results.items()):
    model = get(mid)
    curve = result.get("trajectory")
    if curve:
        key = next(k for k in model.outputs if k in curve[0])
        ax.plot([r[model.axis] for r in curve], [r[key] for r in curve])
        ax.set_xlabel(model.axis, fontsize=8)
        ax.set_ylabel(f"{key} ({model.outputs[key]})", fontsize=7)
    else:
        ax.text(.5,.5,"Scalar screen; inspect result",ha="center",va="center",fontsize=8)
    ax.set_title(mid,fontsize=9)
    ax.grid(alpha=.2)
fig.suptitle("Synthetic examples · separate models · no experimental validation")
fig.tight_layout(rect=(0,0,1,.97))
plt.show()


## 3. Read one model from inputs to result

Change `model_id` to inspect any family. Separate parameters describing material properties from those describing equipment or recipe conditions. A missing input must be measured, sourced or declared as an assumption; the model does not silently invent it.


In [ ]:
model_id = "crystal.cooling_moments"
print(json.dumps(get(model_id).describe(), indent=2))
print(json.dumps(cases[model_id], indent=2))
print({k:v for k,v in results[model_id].items() if k != "trajectory"})


## 4. Compare donor and receiving equipment

This filter example changes only area, holding the declared slurry/resistance assumptions fixed. It is a conditional calculation, not evidence that those properties transfer unchanged. Inspect incomplete outcomes and changed inputs.


In [ ]:
mid = "isolation.cake_filter"
receiver = deepcopy(cases[mid])
receiver["filter_area_m2"] *= 0.8
transfer = compare(mid, cases[mid], receiver, metrics=["completion_time_s"])
print("Changes:", transfer["input_changes"])
print("Receiver minus donor:", transfer["receiver_minus_donor"])
print("Boundary:", transfer["warning"])


## 5. Fit an uncertain parameter and check a holdout

Here the observations are generated from an analytic first-order milling equation. This teaches calibration; it is not experimental milling data. Only observations up to 900 s enter the fit. Later observations test prediction conditional on the same synthetic mechanism.

The same helper supports other declared time/pressure curves, one parameter at a time. Scalar models need a different multi-experiment fitting formulation. A nonzero local sensitivity does not prove general identifiability.


In [ ]:
observations = [{"coordinate":float(t), "value":float(.8*np.exp(-.002*t)), "source_ref":"synthetic-analytic-mill"}
                for t in np.arange(0.,1801.,90.)]
guess = deepcopy(cases["powder.two_bin_mill"])
guess["breakage_rate_s_inv"] = .001
fit = fit_parameter("powder.two_bin_mill", guess, parameter="breakage_rate_s_inv",
                    bounds=[.0001,.005], observable="coarse_mass_fraction",
                    observations=observations, training_until=900.)
print({k:fit[k] for k in ["estimate","parameter_unit","training_rmse","holdout_rmse","identifiability"]})


## 6. Test sensitivity, retaining failures

Try an unsupported area to see that failed scenarios stay visible. These are declared hypotheses, not probabilities or calibrated uncertainty intervals. Do not label the range a 95% confidence interval.


In [ ]:
sweep = parameter_scenarios("isolation.cake_filter", cases["isolation.cake_filter"],
                            parameter="filter_area_m2", values=[-0.01, 0.01, 0.02, 0.04],
                            metrics=["completion_time_s"])
for row in sweep["rows"]:
    print(row)


## 7. Connect crystallisation, filtration, washing and drying

This flowsheet maps actual model outputs between units. It includes seed API in the starting inventory and separately accounts for dissolved mother-liquor API and wash dissolution loss. It reports crystal-derived isolated API only. It does not predict reprecipitation, final assay purity or solvent-species changes.

Retained liquid holdup and density below are explicit assumptions. An incomplete upstream step blocks later steps rather than fabricating yield.


In [ ]:
train = purification_train(crystal=cases["crystal.cooling_moments"],
                           filtration=cases["isolation.cake_filter"], wash=cases["isolation.cake_wash"],
                           dryer=cases["isolation.cake_dryer"], retained_liquid_m3=.00005,
                           liquid_density_kg_m3=800.)
print(json.dumps(train, indent=2))


## 8. Reuse the models we already have

The library examples CLI also runs the original batch reactor, Sulfolobus/Pichia characterisation and primary-drying functions directly. PenSim/SMPL remain behind the existing pinned external worker and are not silently replaced if that environment is unavailable.

**Do now:**

- Chemical synthesis: change feed rate; inspect inventory and temperature/volume stop conditions.
- Crystallisation: change cooling duration and seed size; distinguish crystal mass from mean size.
- Isolation: change filter area, wash volume and drying coefficient; identify yield/time/solvent tradeoffs.
- Powders: change holdup or effective breakage/aggregation rates; distinguish mean behaviour from full distributions.
- Tablets: inspect pressure-density, coating balance and dissolution separately; explain why their connection needs data.
- Biologics downstream: vary product sieving; explain product loss while clearing impurity.
- Utilities: test a gas load that prevents the vacuum target and a viscosity that invalidates turbulent mixing.

For each family write: **What is conserved? What is fitted? What changes at the receiver? What experiment could falsify the prediction?**

Customer readiness requires product-specific calibration and independent receiving-site prediction checks. See [the commercial evidence gates](LIBRARY.md#commercial-readiness-gates). Neural hybrid corrections and calibrated uncertainty remain separate development work.
